# Image denoising

Apply ANTsTorch's non-local means denoiser to a reproducible synthetic image
with additive Gaussian noise. This example needs no downloads or pretrained
weights and checks both image geometry and error against the known clean image.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
rng = np.random.default_rng(42)
xx, yy = np.meshgrid(np.linspace(-1, 1, 48), np.linspace(-1, 1, 40), indexing="ij")
clean = (40 + 30 * np.exp(-5 * (xx**2 + yy**2))).astype("float32")
noisy = (clean + rng.normal(0, 4, clean.shape)).astype("float32")
image = ants.from_numpy(noisy, spacing=(1.2, 1.5), origin=(10., -5.))
denoised = antstorch.denoise_image(
    image, noise_model="Gaussian", p=1, r=2, shrink_factor=1, device=device,
)
output = denoised.numpy()
assert output.shape == clean.shape
assert np.isfinite(output).all()
np.testing.assert_allclose(denoised.spacing, image.spacing)
np.testing.assert_allclose(denoised.origin, image.origin)
np.testing.assert_allclose(denoised.direction, image.direction)
noisy_rmse = float(np.sqrt(np.mean((noisy - clean)**2)))
denoised_rmse = float(np.sqrt(np.mean((output - clean)**2)))
assert denoised_rmse < noisy_rmse
print(f"RMSE against clean image: {noisy_rmse:.3f} -> {denoised_rmse:.3f}")

## Compare signal, noise and residual
The Gaussian model matches the simulated noise. Magnitude MRI may call for a
Rician model instead. Patch radius `p` and search radius `r` affect smoothing,
detail preservation and computation time; tune them for your data.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
for ax, array, title in zip(axes, [clean, noisy, output, noisy - output],
                            ["Clean", "Noisy", "Denoised", "Removed residual"]):
    view = ax.imshow(array.T, origin="lower", cmap="gray",
                     vmin=None if title == "Removed residual" else clean.min(),
                     vmax=None if title == "Removed residual" else clean.max())
    ax.set_title(title); ax.axis("off")
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

For a real image, replace the synthetic input with `ants.image_read(...)` and
inspect edges and small structures after filtering. A low residual variance
alone does not establish preserved anatomy.